# 01 — Feature Extraction Pipeline
## Sensibilidad Temporal de Descriptores de Audio Artesanales

Este notebook implementa:
1. Configuración del entorno (Google Colab)
2. Descarga automática de datasets (GTZAN, FMA-small, MagnaTagATune, IRMAS)
3. Carga y verificación de datasets
4. Extracción de features multi-escala (200ms, 2s, 5s) — 7 descriptores × 3 escalas
5. Cache de features extraídos en Google Drive

**Salida:** Archivos `.npy` con features, labels y splits por dataset en `FEATURES_ROOT`.

**Ejecución:** Google Colab (GPU no necesaria, pero recomendada por RAM)

## 0. Setup & Configuration

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q librosa scikit-learn tqdm soundfile

# Clone the repo to get the src/ modules
import os
REPO_URL = "https://github.com/Gabrieleeh32159/my_paper.git"
REPO_DIR = "/content/my_paper"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

In [ ]:
import os
import sys
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# === CONFIGURATION ===
REPO_DIR = Path('/content/my_paper')

# Google Drive paths for data persistence
DRIVE_ROOT = Path('/content/drive/MyDrive/tsi_experiments')
DATA_ROOT = DRIVE_ROOT / 'data'
FEATURES_ROOT = DRIVE_ROOT / 'features'

# Create output directories
FEATURES_ROOT.mkdir(parents=True, exist_ok=True)

# Add repo's experiments/ to path so `from src.X import ...` works
sys.path.insert(0, str(REPO_DIR / 'experiments'))

# Dataset paths (data lives on Drive)
DATASET_PATHS = {
    'gtzan': DATA_ROOT / 'gtzan',
    'fma_small': DATA_ROOT,  # FMA expects fma_small/ and fma_metadata/ under root
    'mtat': DATA_ROOT / 'magnatagatune',
    'irmas': DATA_ROOT / 'irmas',
}

# Random seed for reproducibility
SEED = 42
np.random.seed(SEED)

print(f"Repo source: {REPO_DIR}")
print(f"Drive root:  {DRIVE_ROOT}")
print(f"Data root:   {DATA_ROOT}")
print(f"Features:    {FEATURES_ROOT}")

In [ ]:
%cd {REPO_DIR}/experiments

from src.features import FEATURE_DIMS, SCALES, TRACK_DIM
from src.data_loader import get_dataset

print("Modules loaded.")
print(f"Feature dimensions per frame: {sum(FEATURE_DIMS.values())} ({FEATURE_DIMS})")
print(f"Track vector dim per scale: {TRACK_DIM}")
print(f"Temporal scales: {SCALES}")

## 1. Data Download & Verification

In [ ]:
# Verify available datasets
print("=== Dataset Availability ===")
for name, path in DATASET_PATHS.items():
    exists = path.exists()
    print(f"  {name:12s}: {'✅' if exists else '❌'} {path}")
    if exists:
        audio_exts = {'.mp3', '.wav', '.au', '.ogg'}
        n_files = sum(1 for f in path.rglob('*') if f.suffix in audio_exts)
        print(f"               {n_files} audio files found")

In [ ]:
# === AUTO-DOWNLOAD DATASETS ===

import subprocess
import tarfile
import shutil


def download_gtzan(dest_root):
    """
    Download GTZAN dataset.
    Expected structure: dest_root/genres_original/{genre}/*.wav
    """
    dest_root = Path(dest_root)
    genres_dir = dest_root / 'genres_original'
    if genres_dir.exists() and any(genres_dir.iterdir()):
        print("  [gtzan] Already exists, skipping.")
        return True

    dest_root.mkdir(parents=True, exist_ok=True)
    tar_path = dest_root / 'genres.tar.gz'

    url = "http://opihi.cs.uvic.ca/sound/genres.tar.gz"
    print(f"  [gtzan] Downloading from {url}...")

    !wget -q --show-progress -O {tar_path} {url} 2>&1 || true

    if not tar_path.exists() or tar_path.stat().st_size < 1000000:
        print("  [gtzan] Primary source failed. Trying Kaggle...")
        !pip install -q kagglehub 2>/dev/null
        try:
            import kagglehub
            kaggle_path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")
            kaggle_path = Path(kaggle_path)
            src_genres = kaggle_path / 'Data' / 'genres_original'
            if src_genres.exists():
                shutil.copytree(src_genres, genres_dir, dirs_exist_ok=True)
                print("  [gtzan] Downloaded via Kaggle.")
                return True
        except Exception as e:
            print(f"  [gtzan] Kaggle fallback failed: {e}")
            return False

    if tar_path.exists() and tar_path.stat().st_size > 1000000:
        print("  [gtzan] Extracting...")
        with tarfile.open(tar_path, 'r:gz') as tar:
            tar.extractall(path=str(dest_root))
        tar_path.unlink()

        if (dest_root / 'genres').exists() and not genres_dir.exists():
            (dest_root / 'genres').rename(genres_dir)
        print("  [gtzan] Done.")
        return True

    return False


def download_fma_small(dest_root):
    """
    Download FMA-small dataset (audio + metadata).
    Expected: dest_root/fma_small/ and dest_root/fma_metadata/
    """
    dest_root = Path(dest_root)
    audio_dir = dest_root / 'fma_small'
    metadata_dir = dest_root / 'fma_metadata'

    if not audio_dir.exists() or not any(audio_dir.iterdir()):
        dest_root.mkdir(parents=True, exist_ok=True)
        zip_path = dest_root / 'fma_small.zip'

        url = "https://os.unil.cloud.switch.ch/fma/fma_small.zip"
        print(f"  [fma_small] Downloading audio ({url})...")
        print("  [fma_small] This is ~7.2 GB, may take a while...")

        !wget -q --show-progress -O {zip_path} {url}

        if zip_path.exists() and zip_path.stat().st_size > 1000000:
            print("  [fma_small] Extracting audio...")
            !unzip -q -o {zip_path} -d {dest_root}
            zip_path.unlink()
            print("  [fma_small] Audio extracted.")
        else:
            print("  [fma_small] Audio download failed!")
            return False
    else:
        print("  [fma_small] Audio already exists.")

    if not metadata_dir.exists() or not (metadata_dir / 'tracks.csv').exists():
        zip_path = dest_root / 'fma_metadata.zip'

        url = "https://os.unil.cloud.switch.ch/fma/fma_metadata.zip"
        print(f"  [fma_small] Downloading metadata ({url})...")

        !wget -q --show-progress -O {zip_path} {url}

        if zip_path.exists() and zip_path.stat().st_size > 100000:
            print("  [fma_small] Extracting metadata...")
            !unzip -q -o {zip_path} -d {dest_root}
            zip_path.unlink()
            print("  [fma_small] Metadata extracted.")
        else:
            print("  [fma_small] Metadata download failed!")
            return False
    else:
        print("  [fma_small] Metadata already exists.")

    return True


def download_magnatagatune(dest_root):
    """
    Download MagnaTagATune dataset.
    Expected: dest_root/mp3/, dest_root/annotations_final.csv, dest_root/split/
    """
    dest_root = Path(dest_root)
    dest_root.mkdir(parents=True, exist_ok=True)

    annotations_file = dest_root / 'annotations_final.csv'
    if not annotations_file.exists():
        url = "https://mirg.city.ac.uk/datasets/magnatagatune/annotations_final.csv"
        print(f"  [mtat] Downloading annotations...")
        !wget -q --show-progress -O {annotations_file} {url}

    mp3_dir = dest_root / 'mp3'
    if not mp3_dir.exists() or not any(mp3_dir.iterdir()):
        for part in range(1, 4):
            zip_name = f"mp3.zip.{part:03d}"
            zip_path = dest_root / zip_name
            url = f"https://mirg.city.ac.uk/datasets/magnatagatune/{zip_name}"
            print(f"  [mtat] Downloading audio part {part}/3...")
            !wget -q --show-progress -O {zip_path} {url}

        print("  [mtat] Combining and extracting audio parts...")
        !cd {dest_root} && cat mp3.zip.001 mp3.zip.002 mp3.zip.003 > mp3_combined.zip
        !cd {dest_root} && unzip -q -o mp3_combined.zip

        for part in range(1, 4):
            (dest_root / f"mp3.zip.{part:03d}").unlink(missing_ok=True)
        (dest_root / 'mp3_combined.zip').unlink(missing_ok=True)
        print("  [mtat] Audio extracted.")
    else:
        print("  [mtat] Audio already exists.")

    split_dir = dest_root / 'split'
    if not split_dir.exists():
        split_dir.mkdir(parents=True, exist_ok=True)
        print("  [mtat] Downloading split files...")

        base_url = "https://raw.githubusercontent.com/keunwoochoi/magnatagatune-list/master"
        for split_file in ['train_list.txt', 'valid_list.txt', 'test_list.txt']:
            url = f"{base_url}/{split_file}"
            local_name = split_file.replace('_list', '')
            !wget -q -O {split_dir / local_name} {url}

        for fname in ['train.txt', 'valid.txt', 'test.txt']:
            fpath = split_dir / fname
            if fpath.exists():
                lines = fpath.read_text().strip().split('\n')
                clip_ids = []
                for line in lines:
                    parts = line.strip().split('\t')
                    if len(parts) >= 2:
                        clip_ids.append(parts[1])
                    else:
                        clip_ids.append(parts[0])
                fpath.write_text('\n'.join(clip_ids))
        print("  [mtat] Split files ready.")
    else:
        print("  [mtat] Split files already exist.")

    return True


def download_irmas(dest_root):
    """
    Download IRMAS dataset.
    Expected: dest_root/IRMAS-TrainingData/ and dest_root/IRMAS-TestingData-Part1/
    """
    dest_root = Path(dest_root)
    dest_root.mkdir(parents=True, exist_ok=True)

    train_dir = dest_root / 'IRMAS-TrainingData'
    test_dir = dest_root / 'IRMAS-TestingData-Part1'

    if not train_dir.exists() or not any(train_dir.iterdir()):
        zip_path = dest_root / 'IRMAS-TrainingData.zip'
        url = "https://zenodo.org/record/1290750/files/IRMAS-TrainingData.zip"
        print(f"  [irmas] Downloading training data...")
        !wget -q --show-progress -O {zip_path} {url}

        if zip_path.exists() and zip_path.stat().st_size > 1000000:
            print("  [irmas] Extracting training data...")
            !unzip -q -o {zip_path} -d {dest_root}
            zip_path.unlink()
        else:
            print("  [irmas] Training data download failed!")
            return False
    else:
        print("  [irmas] Training data already exists.")

    if not test_dir.exists():
        zip_path = dest_root / 'IRMAS-TestingData-Part1.zip'
        url = "https://zenodo.org/record/1290750/files/IRMAS-TestingData-Part1.zip"
        print(f"  [irmas] Downloading testing data...")
        !wget -q --show-progress -O {zip_path} {url}

        if zip_path.exists() and zip_path.stat().st_size > 100000:
            print("  [irmas] Extracting testing data...")
            !unzip -q -o {zip_path} -d {dest_root}
            zip_path.unlink()
        else:
            print("  [irmas] Testing data download failed!")
            return False
    else:
        print("  [irmas] Testing data already exists.")

    return True


# === RUN DOWNLOADS ===
print("=" * 60)
print("CHECKING AND DOWNLOADING MISSING DATASETS")
print("=" * 60)

download_status = {}

# GTZAN
print("\n[1/4] GTZAN Dataset (~1.2 GB)")
gtzan_path = DATASET_PATHS['gtzan']
if not gtzan_path.exists() or not any(gtzan_path.iterdir()):
    download_status['gtzan'] = download_gtzan(gtzan_path)
else:
    print("  [gtzan] Already available.")
    download_status['gtzan'] = True

# FMA-small
print("\n[2/4] FMA-Small Dataset (~7.2 GB audio + metadata)")
fma_path = DATASET_PATHS['fma_small']
fma_audio = fma_path / 'fma_small'
fma_meta = fma_path / 'fma_metadata'
if not fma_audio.exists() or not fma_meta.exists():
    download_status['fma_small'] = download_fma_small(fma_path)
else:
    print("  [fma_small] Already available.")
    download_status['fma_small'] = True

# MagnaTagATune
print("\n[3/4] MagnaTagATune Dataset (~3.3 GB)")
mtat_path = DATASET_PATHS['mtat']
if not mtat_path.exists() or not (mtat_path / 'annotations_final.csv').exists():
    download_status['mtat'] = download_magnatagatune(mtat_path)
else:
    print("  [mtat] Already available.")
    download_status['mtat'] = True

# IRMAS
print("\n[4/4] IRMAS Dataset (~3.1 GB)")
irmas_path = DATASET_PATHS['irmas']
if not irmas_path.exists() or not (irmas_path / 'IRMAS-TrainingData').exists():
    download_status['irmas'] = download_irmas(irmas_path)
else:
    print("  [irmas] Already available.")
    download_status['irmas'] = True

# Summary
print("\n" + "=" * 60)
print("DOWNLOAD SUMMARY")
print("=" * 60)
for name, status in download_status.items():
    emoji = '✅' if status else '❌'
    print(f"  {name:12s}: {emoji}")

## 2. Dataset Loading

In [ ]:
# Load datasets
datasets = {}

try:
    datasets['fma_small'] = get_dataset('fma_small', str(DATASET_PATHS['fma_small']))
    print(f"FMA-small: {len(datasets['fma_small'])} tracks, {datasets['fma_small'].n_classes} classes")
except Exception as e:
    print(f"FMA-small failed: {e}")

try:
    datasets['gtzan'] = get_dataset('gtzan', str(DATASET_PATHS['gtzan']))
    print(f"GTZAN: {len(datasets['gtzan'])} tracks, {datasets['gtzan'].n_classes} classes")
except Exception as e:
    print(f"GTZAN failed: {e}")

try:
    datasets['mtat'] = get_dataset('mtat', str(DATASET_PATHS['mtat']))
    print(f"MTAT: {len(datasets['mtat'])} clips, {datasets['mtat'].n_classes} tags")
except Exception as e:
    print(f"MTAT failed: {e}")

try:
    datasets['irmas'] = get_dataset('irmas', str(DATASET_PATHS['irmas']))
    print(f"IRMAS: {len(datasets['irmas'])} fragments, {datasets['irmas'].n_classes} instruments")
except Exception as e:
    print(f"IRMAS failed: {e}")

print(f"\n=== Successfully loaded: {list(datasets.keys())} ===")

## 3. Multi-Scale Feature Extraction

Extrae 7 descriptores × 3 escalas para todos los tracks. Los resultados se cachean en Drive.

**Optimización:** Se computan los features a nivel de frame una sola vez por track
y luego se segmentan por ventana temporal, reduciendo ~170× las llamadas a librosa.

In [ ]:
import gc
import json
import subprocess
import resource
import numpy as np
from pathlib import Path
from tqdm import tqdm

# Limit BLAS threads to 1 (prevents threading crashes in C libs)
try:
    from threadpoolctl import threadpool_limits
    threadpool_limits(1)
    print("BLAS threads limited to 1 via threadpoolctl")
except ImportError:
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    print("BLAS thread env vars set to 1")

print("Feature extraction: sequential mode (optimized, single-pass features)")

In [ ]:
from src.features import (
    SCALES, HOP_LENGTH, N_FFT, FEATURE_DIMS,
    extract_frame_features, aggregate_window
)


def _safe_load_audio(filepath, sr=16000):
    """
    Load audio without librosa.load (avoids libsndfile/audioread crashes).
    Uses soundfile for WAV/FLAC, ffmpeg subprocess for MP3/other.
    """
    filepath = str(filepath)

    try:
        import soundfile as sf
        y, orig_sr = sf.read(filepath, dtype='float32')
        if y.ndim > 1:
            y = y.mean(axis=1)
        if orig_sr != sr:
            import librosa
            y = librosa.resample(y, orig_sr=orig_sr, target_sr=sr)
    except Exception:
        cmd = [
            'ffmpeg', '-i', filepath,
            '-f', 'f32le', '-acodec', 'pcm_f32le',
            '-ar', str(sr), '-ac', '1',
            '-v', 'error', '-'
        ]
        proc = subprocess.run(cmd, capture_output=True, timeout=120)
        if proc.returncode != 0:
            raise RuntimeError(f"ffmpeg error: {proc.stderr.decode()[:200]}")
        y = np.frombuffer(proc.stdout, dtype=np.float32).copy()
        del proc
        if len(y) == 0:
            raise RuntimeError("Empty audio")

    # Simple silence trim
    amp = np.abs(y)
    threshold = np.max(amp) * 0.01 if np.max(amp) > 0 else 0
    above = np.where(amp > threshold)[0]
    if len(above) > 0:
        y = y[above[0]:above[-1] + 1]
    del amp

    return y


def _extract_multiscale_fast(y, sr=16000):
    """
    Extract 7 descriptors x 3 scales in a SINGLE pass.

    Computes frame-level features ONCE on the full track, then slices
    the frame matrices into temporal windows for aggregation.

    Result: {scale_name: 192-d vector}.
    """
    # Compute all frame-level features ONCE
    frame_feats = extract_frame_features(y, sr)
    n_frames = frame_feats['mfcc'].shape[1]

    result = {}
    for scale_name, window_sec in SCALES.items():
        window_samples = int(window_sec * sr)
        frames_per_window = max(1, window_samples // HOP_LENGTH)

        n_windows = n_frames // frames_per_window
        if n_windows == 0:
            n_windows = 1
            frames_per_window = n_frames

        window_vectors = []
        for w_idx in range(n_windows):
            start = w_idx * frames_per_window
            end = min(start + frames_per_window, n_frames)
            if end - start < 2:
                continue

            window_feats = {
                name: feat[:, start:end]
                for name, feat in frame_feats.items()
            }
            agg = aggregate_window(window_feats)
            window_vectors.append(agg)

        if not window_vectors:
            agg = aggregate_window(frame_feats)
            window_vectors = [agg]

        window_matrix = np.stack(window_vectors)
        track_mean = np.mean(window_matrix, axis=0)
        track_std = np.std(window_matrix, axis=0)
        result[scale_name] = np.concatenate([track_mean, track_std])

    del frame_feats
    return result


def _get_rss_mb():
    """Get current RSS memory usage in MB."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024 * 1024)

In [ ]:
def extract_dataset_features(dataset_name, dataset, output_dir, sr=16000):
    """
    Sequential incremental feature extraction (crash-resilient).
    - Single-pass feature computation
    - Audio loading via soundfile/ffmpeg
    - Checkpoints every 100 tracks
    - Aggressive GC every 25 tracks
    - Memory monitoring
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    n_tracks = len(dataset)

    # File paths
    short_path   = output_dir / f"{dataset_name}_short.npy"
    medium_path  = output_dir / f"{dataset_name}_medium.npy"
    long_path    = output_dir / f"{dataset_name}_long.npy"
    indices_path = output_dir / f"{dataset_name}_indices.npy"
    errors_path  = output_dir / f"{dataset_name}_errors.json"

    # Load existing cached data
    done_set = set()
    cached_s, cached_m, cached_l, cached_idx = None, None, None, None

    if short_path.exists() and indices_path.exists():
        try:
            cached_s   = np.load(short_path)
            cached_m   = np.load(medium_path)
            cached_l   = np.load(long_path)
            cached_idx = np.load(indices_path)
            done_set   = set(cached_idx.tolist())
            print(f"  [{dataset_name}] Loaded {len(cached_idx)} cached tracks")
        except Exception as e:
            print(f"  [{dataset_name}] Cache corrupted, starting fresh: {e}")
            cached_s = cached_m = cached_l = cached_idx = None

    # Load previous errors
    all_errors = []
    if errors_path.exists():
        try:
            with open(errors_path) as f:
                all_errors = json.load(f)
        except Exception:
            pass
    error_indices = {e[0] for e in all_errors}

    # Determine missing tracks
    missing = sorted(set(range(n_tracks)) - done_set - error_indices)
    print(f"  [{dataset_name}] Status: {len(done_set)} OK, {len(error_indices)} errors, "
          f"{len(missing)} remaining (total: {n_tracks})")
    print(f"  [{dataset_name}] RSS memory: {_get_rss_mb():.0f} MB")

    if not missing:
        if cached_s is not None:
            order = np.argsort(cached_idx)
            fi = cached_idx[order]
            fs, fm, fl = cached_s[order], cached_m[order], cached_l[order]
            labels = np.array([dataset.get_label(int(i)) for i in fi], dtype=object)
            splits = np.array([dataset.get_split(int(i)) for i in fi])
            np.save(output_dir / f"{dataset_name}_labels.npy", labels)
            np.save(output_dir / f"{dataset_name}_splits.npy", splits)
            print(f"  [{dataset_name}] Complete: {len(fi)} tracks x 192-d x 3 scales")
            return {'short': fs, 'medium': fm, 'long': fl}, labels, splits
        else:
            raise RuntimeError(f"No data found for {dataset_name}!")

    # Extract missing tracks
    new_s, new_m, new_l, new_idx = [], [], [], []
    new_errors = []
    checkpoint_every = 100
    gc_every = 25

    pbar = tqdm(missing, desc=f"[{dataset_name}]", unit="track")
    for i, track_idx in enumerate(pbar):
        filepath = str(dataset.get_audio_path(track_idx))
        try:
            y = _safe_load_audio(filepath, sr=sr)
            if len(y) < sr:
                new_errors.append([int(track_idx), filepath, "too short"])
                continue
            feats = _extract_multiscale_fast(y, sr=sr)
            new_s.append(feats['short'])
            new_m.append(feats['medium'])
            new_l.append(feats['long'])
            new_idx.append(int(track_idx))
            del y, feats
        except Exception as e:
            new_errors.append([int(track_idx), filepath, str(e)[:200]])
            continue

        if (i + 1) % gc_every == 0:
            gc.collect()

        if (i + 1) % 200 == 0:
            pbar.set_postfix(
                ok=len(done_set) + len(new_idx),
                err=len(all_errors) + len(new_errors),
                mem=f"{_get_rss_mb():.0f}MB"
            )

        if len(new_idx) > 0 and len(new_idx) % checkpoint_every == 0:
            pbar.set_postfix(saving="checkpoint")
            _save_feature_checkpoint(
                dataset_name, output_dir,
                cached_s, cached_m, cached_l, cached_idx,
                new_s, new_m, new_l, new_idx,
                all_errors + new_errors
            )
            pbar.set_postfix(saved=len(done_set) + len(new_idx))

    # Final save
    merged_errors = all_errors + new_errors
    seen = set()
    unique_errors = [e for e in merged_errors if e[0] not in seen and not seen.add(e[0])]

    parts_s, parts_m, parts_l, parts_i = [], [], [], []
    if cached_s is not None:
        parts_s.append(cached_s); parts_m.append(cached_m)
        parts_l.append(cached_l); parts_i.append(cached_idx)
    if new_idx:
        parts_s.append(np.stack(new_s)); parts_m.append(np.stack(new_m))
        parts_l.append(np.stack(new_l)); parts_i.append(np.array(new_idx))

    if not parts_i:
        raise RuntimeError(f"No data for {dataset_name}!")

    fs = np.concatenate(parts_s); fm = np.concatenate(parts_m)
    fl = np.concatenate(parts_l); fi = np.concatenate(parts_i)

    _, upos = np.unique(fi, return_index=True)
    fi, fs, fm, fl = fi[upos], fs[upos], fm[upos], fl[upos]
    order = np.argsort(fi)
    fi, fs, fm, fl = fi[order], fs[order], fm[order], fl[order]

    labels = np.array([dataset.get_label(int(i)) for i in fi], dtype=object)
    splits = np.array([dataset.get_split(int(i)) for i in fi])

    np.save(short_path,  fs)
    np.save(medium_path, fm)
    np.save(long_path,   fl)
    np.save(indices_path, fi)
    np.save(output_dir / f"{dataset_name}_labels.npy", labels)
    np.save(output_dir / f"{dataset_name}_splits.npy", splits)

    if unique_errors:
        with open(errors_path, 'w') as f:
            json.dump(unique_errors, f, indent=2, default=str)

    n_new = len(new_idx)
    print(f"  [{dataset_name}] {len(fi)} tracks (+{n_new} new), "
          f"{len(unique_errors)} errors x 192-d x 3 scales")
    print(f"  [{dataset_name}] Final RSS: {_get_rss_mb():.0f} MB")

    del new_s, new_m, new_l, parts_s, parts_m, parts_l
    del cached_s, cached_m, cached_l
    gc.collect()

    return {'short': fs, 'medium': fm, 'long': fl}, labels, splits


def _save_feature_checkpoint(dataset_name, output_dir,
                             cached_s, cached_m, cached_l, cached_idx,
                             new_s, new_m, new_l, new_idx, errors):
    """Lightweight checkpoint: save feature arrays + indices only."""
    parts_s, parts_m, parts_l, parts_i = [], [], [], []
    if cached_s is not None:
        parts_s.append(cached_s); parts_m.append(cached_m)
        parts_l.append(cached_l); parts_i.append(cached_idx)
    if new_idx:
        parts_s.append(np.stack(new_s)); parts_m.append(np.stack(new_m))
        parts_l.append(np.stack(new_l)); parts_i.append(np.array(new_idx))

    if not parts_i:
        return

    fs = np.concatenate(parts_s); fm = np.concatenate(parts_m)
    fl = np.concatenate(parts_l); fi = np.concatenate(parts_i)

    _, upos = np.unique(fi, return_index=True)
    fi, fs, fm, fl = fi[upos], fs[upos], fm[upos], fl[upos]
    order = np.argsort(fi)
    fi, fs, fm, fl = fi[order], fs[order], fm[order], fl[order]

    np.save(output_dir / f"{dataset_name}_short.npy",   fs)
    np.save(output_dir / f"{dataset_name}_medium.npy",  fm)
    np.save(output_dir / f"{dataset_name}_long.npy",    fl)
    np.save(output_dir / f"{dataset_name}_indices.npy", fi)

    if errors:
        seen = set()
        unique = [e for e in errors if e[0] not in seen and not seen.add(e[0])]
        with open(output_dir / f"{dataset_name}_errors.json", 'w') as f:
            json.dump(unique, f, indent=2, default=str)

    del fs, fm, fl, fi, parts_s, parts_m, parts_l, parts_i
    gc.collect()

In [ ]:
# Extract features for all available datasets
all_features = {}
all_labels = {}
all_splits = {}

for name, ds in datasets.items():
    print(f"\nProcessing {name}...")
    feats, labels, splits = extract_dataset_features(name, ds, FEATURES_ROOT)
    all_features[name] = feats
    all_labels[name] = labels
    all_splits[name] = splits
    gc.collect()

print("\n=== Feature Extraction Complete ===")
for name in all_features:
    n_tracks = all_features[name]['short'].shape[0]
    print(f"  {name}: {n_tracks} tracks x 192-d x 3 scales")
    print(f"    Files saved: {FEATURES_ROOT / name}_*.npy")

## 4. Verification

Verifica que los archivos de features estén correctos antes de continuar con el análisis.

In [ ]:
# Verify saved features
print("=== Saved Feature Files ===")
for f in sorted(FEATURES_ROOT.glob('*.npy')):
    arr = np.load(f, allow_pickle=True)
    print(f"  {f.name:40s} shape={str(arr.shape):20s} dtype={arr.dtype}")

print("\n=== Error Files ===")
for f in sorted(FEATURES_ROOT.glob('*.json')):
    with open(f) as fh:
        errors = json.load(fh)
    print(f"  {f.name}: {len(errors)} errors")

print("\n Feature extraction complete.")
print(f" Results saved to: {FEATURES_ROOT}")
print(" Continue with notebook 02_tsi_analysis.ipynb for TSI computation and analysis.")